In [1]:
import os, sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT)); os.chdir(PROJECT_ROOT)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import warnings; warnings.filterwarnings("ignore")
%matplotlib inline
print("Raiz do projeto:", PROJECT_ROOT)

Raiz do projeto: C:\Users\josepires\Downloads\Projeto  IA III\IA3_Projeto-José


# 5. Deployment simulado: sistema multiagente (CRISP-ML)

Este notebook **chama todos os modelos e agentes** guardados no projeto:
- **Classificadores** em `models/` (7 de scikit-learn + MLP TensorFlow) e o **modelo campeão** do manifesto;
- **LLM generativo** (Transformers, configurado em `models/llm_config.json`) com *fallback* offline;
- **Agentes**: Classificador → Analista → Relator (LLM) → Notificador (Telegram), coordenados por um Orquestrador;
- **Skills** em `skills/*/SKILL.md`;
- **Segunda versão** com **CrewAI** e fluxo **n8n** (`n8n/`).

A versão equivalente em Python é `run_system.py`.

## 5.1. Todos os modelos treinados em `models/`

In [2]:
import json, pandas as pd
from src import evaluate_models as ev
df_eval, scores, y_te = ev.avaliar_modelos_guardados()
print("LLM config:", json.load(open("models/llm_config.json", encoding="utf-8"))["model_id"])
df_eval[["Modelo", "Tipo", "Precision Fraude (%)", "Recall Fraude (%)", "F1 Fraude (%)", "PR-AUC (%)"]]

LLM config: Qwen/Qwen2.5-0.5B-Instruct


,Modelo,Tipo,Precision Fraude (%),Recall Fraude (%),F1 Fraude (%),PR-AUC (%)
0,Random Forest,Machine Learning,98.15,71.62,82.81,81.41
1,TensorFlow Neural Net,Deep Learning,11.23,85.14,19.84,79.40
2,KNN,Machine Learning,76.71,75.68,76.19,79.33
3,Logistic Regression,Machine Learning,2.68,91.89,5.20,75.92
4,Decision Tree,Machine Learning,11.11,77.03,19.42,70.99
5,SVM,Machine Learning,52.43,72.97,61.02,69.06
6,Gradient Boosting,Machine Learning,55.21,71.62,62.35,38.34
7,Naive Bayes,Machine Learning,5.03,79.73,9.46,6.19


## 5.2. Skills dos agentes

In [3]:
from src.agents.skills import carregar_skills
for nome, s in carregar_skills().items():
    print(f"- {nome}: {s['descricao']}  [{s['ficheiro']}]")

- alerta_telegram: Envia por Telegram alertas de fraude (CRÍTICO/ALTO) e o dashboard diário; em modo simulação grava a mensagem em reports/telegram_outbox.jsonl.  [skills\alerta_telegram\SKILL.md]
- analisar_lote: Analisa um lote de transações classificadas e devolve estatísticas do dia, valor em risco, hora de pico e alertas ordenados por gravidade.  [skills\analisar_lote\SKILL.md]
- classificar_transacao: Classifica uma transação de cartão (V1..V28, Amount, Time/Hour) como Fraude ou Legítima com o modelo campeão e devolve a probabilidade.  [skills\classificar_transacao\SKILL.md]
- relatorio_diario: Gera o relatório diário de fraude em português a partir dos factos do lote, usando um LLM (Transformers) com guardrail de números.  [skills\relatorio_diario\SKILL.md]


## 5.3. Orquestrador e agentes
O relator usa Transformers se houver internet/torch; caso contrário, o *fallback* offline mantém o sistema funcional.

In [4]:
from src.agents.agents import Orquestrador
from src.agents.llm import get_backend
orq = Orquestrador(backend=get_backend("auto"))
print("Backend do relator:", orq.relator.backend.nome)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Backend do relator: transformers:Qwen/Qwen2.5-0.5B-Instruct


## 5.4. Simulação de situações
Três cenários com transações reais do conjunto de teste: **dia normal**, **ataque** (rajada de fraudes) e **deriva de valores**.

In [5]:
from src.agents.simulator import cenario
resultados = {}
for nome in ["dia_normal", "ataque", "deriva_valores"]:
    resultados[nome] = orq.processar_lote(cenario(nome))
    print(f"\n=== {nome} ===")
    for ag, msg in resultados[nome]["trace"].items():
        print(f"  [{ag}] {msg}")


=== dia_normal ===
  [Classificador] classificou 3000 transações (0 alertas)
  [Analista] gerou factos: 0 alertas, 0.0 EUR em risco
  [Relator] relatório gerado com transformers:Qwen/Qwen2.5-0.5B-Instruct
  [Notificador] enviou 0 alertas + dashboard + relatório (simulação)



=== ataque ===
  [Classificador] classificou 3000 transações (28 alertas)
  [Analista] gerou factos: 28 alertas, 2018.57 EUR em risco
  [Relator] relatório gerado com transformers:Qwen/Qwen2.5-0.5B-Instruct
  [Notificador] enviou 5 alertas + dashboard + relatório (simulação)



=== deriva_valores ===
  [Classificador] classificou 3000 transações (0 alertas)
  [Analista] gerou factos: 0 alertas, 0.0 EUR em risco
  [Relator] relatório gerado com transformers:Qwen/Qwen2.5-0.5B-Instruct
  [Notificador] enviou 0 alertas + dashboard + relatório (simulação)


In [6]:
pd.DataFrame({n: {k: v for k, v in r["factos"].items() if not isinstance(v, (dict, list))} for n, r in resultados.items()})

,dia_normal,ataque,deriva_valores
n_transacoes,3000.00,3000.000,3000.00
n_alertas,0.00,28.000,0.00
taxa_alertas_%,0.00,0.933,0.00
valor_total,231028.11,232924.650,1849448.24
valor_em_risco,0.00,2018.570,0.00
prob_media_alertas_%,0.00,89.400,0.00
hora_pico_alertas,NaN,18.000,NaN


### Relatório gerado no cenário de ataque

In [7]:
from IPython.display import Markdown
Markdown(resultados["ataque"]["relatorio"])

# Relatório diário de fraude - 2026-11-21

*Modelo: Random Forest | Gerado por: transformers:Qwen/Qwen2.5-0.5B-Instruct*

## Resumo
### Relatório de Análise de Fraude Bancária

**Transações Analisadas:** 3000
**Alertas de Fraude:** 28 (0.933% das transações)
**Valor Total Movimentado:** 232924.65 euros
**Valor Em Risco:** 2018.57 euros
**Probabilidade Média dos Alertas:** 89.4%
**Hora Com Mais Alertas:** 18h
**Alertas Por Gravidade:** CRÍTICO 16, ALTO 10, MÉDIO 2
**Simulação:** 
- Fraudes Reais: 40
- Detetadas: 28
- Falhas: 12
- Falsos Positivos: 0

## Factos (verificados, não gerados pelo LLM)
| Métrica | Valor |
|---|---|
| n_transacoes | 3000 |
| n_alertas | 28 |
| taxa_alertas_% | 0.933 |
| valor_total | 232924.65 |
| valor_em_risco | 2018.57 |
| prob_media_alertas_% | 89.4 |
| hora_pico_alertas | 18 |
| alertas_por_gravidade | {'CRÍTICO': 16, 'ALTO': 10, 'MÉDIO': 2} |

## Top alertas
- #1558: 100.0% | 94.82 EUR | 19h | CRÍTICO
- #1388: 99.0% | 1.0 EUR | 18h | CRÍTICO
- #110: 99.0% | 4.69 EUR | 20h | CRÍTICO
- #526: 99.0% | 6.74 EUR | 18h | CRÍTICO
- #1647: 98.0% | 0.76 EUR | 17h | CRÍTICO


## 5.5. Telegram (alertas, dashboard e comandos)
Sem `TELEGRAM_BOT_TOKEN`/`TELEGRAM_CHAT_ID` o bot funciona em **modo simulação** e grava em `reports/telegram_outbox.jsonl`.

In [8]:
print("Modo simulação:", orq.notificador.tg.simulacao)
print(orq.responder("/modelo", resultados["ataque"]["factos"]))
print()
print(orq.responder("/dashboard", resultados["ataque"]["factos"]))
print("\nÚltimas mensagens na outbox:")
for linha in open("reports/telegram_outbox.jsonl", encoding="utf-8").read().splitlines()[-2:]:
    print(" ", json.loads(linha)["texto"][:80].replace("\n", " | "))

Modo simulação: True
🤖 Random Forest | PR-AUC 81.41% | Recall 71.62% | limiar 0.5

📊 Dashboard diário de fraude
Transações: 3000 | Alertas: 28 (0.933%)
Valor em risco: 2018.57 EUR de 232924.65 EUR
Hora de pico: 18h | Gravidade: {'CRÍTICO': 16, 'ALTO': 10, 'MÉDIO': 2}
Modelo: Random Forest (PR-AUC 81.41%)
Simulação: 28/40 fraudes detetadas, 0 falsos positivos

Últimas mensagens na outbox:
  📊 Dashboard diário de fraude | Transações: 3000 | Alertas: 0 (0.0%) | Valor em risco
  # Relatório diário de fraude - 2026-11-21 |  | *Modelo: Random Forest | Gerado por: 


## 5.6. Segunda versão do sistema multiagente: CrewAI
Mesmas ferramentas, outra biblioteca. A construção da *Crew* não precisa de LLM; `crew.kickoff()` precisa de um LLM configurado (ex.: Ollama).

In [9]:
from src.agents.crewai_version import construir_crew
crew = construir_crew("ollama/qwen2.5:3b")
print("Agentes:", [a.role for a in crew.agents])
print("Tarefas:", [t.description[:60] + "..." for t in crew.tasks])

Agentes: ['Analista de fraude', 'Relator']
Tarefas: ["Classifica o cenário '{cenario}' e devolve os factos do lote...", 'Com os factos anteriores, escreve o relatório diário com 2 r...']


## 5.7. Versão n8n
O ficheiro `n8n/fraude_workflow.json` importa-se no n8n e usa a API local (`python -m src.api_server`) para o dashboard diário e para classificar transações via *webhook*.

In [10]:
wf = json.load(open("n8n/fraude_workflow.json", encoding="utf-8"))
print([n["name"] for n in wf["nodes"]])

['Todos os dias às 20h', 'Pedir dashboard à API', 'Telegram - Dashboard', 'Webhook - nova transação', 'Classificar (modelo campeão)', 'É fraude?', 'Telegram - Alerta']


## 5.8. Conclusões
- O sistema separa **decisão (ML)**, **análise (regras)** e **linguagem (LLM)**; os números do relatório são sempre os do Analista (*guardrail*).
- No cenário de ataque os alertas sobem de forma visível; no dia normal o recall não é 100% (limitação do modelo, ~72% no teste).
- Limitações: o LLM só corre com internet/torch; o Telegram real exige token; o CrewAI exige um LLM configurado.